# Prefix-cached ACDC on standardized MIB IOI

Independent implementation of the computational optimization described in the supplied research log, Stage J. Its reported **2.35×** speedup used auto-circuit IOI and squared probability distance; the original implementation and trace are unavailable here. Our MIB adaptation uses the official negative IOI logit difference and a documented compatible exact edge-pruning traversal. It is not an exact numerical reproduction of that independent configuration.

The corrected dynamic sender semantics are retained: donor minus **current intervened sender**. No sketching, Hessians, group deletion or MAttr methods are used. Previous experiments and installed environments are preserved.

In [5]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, subprocess
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/acdc_prefix_cache').is_dir())
campaigns=sorted((ROOT/'results/acdc_prefix_cache').glob('*')) if (ROOT/'results/acdc_prefix_cache').exists() else []
RESULTS=campaigns[-1] if campaigns else ROOT/'results/acdc_prefix_cache'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_STAGE_A=True
RUN_STAGE_B=True
RUN_STAGE_C=True
ALLOW_FULL_GRAPH=True
print('Campaign:',RESULTS)
print('Defaults: GPT-2 Small, official MIB IOI, float32, train 16, validation 16, batch 2, seed 42, threshold 0.05.')

def linux_path(path):
    path=Path(path).resolve()
    return '/mnt/'+path.drive[0].lower()+path.as_posix()[2:] if os.name=='nt' else str(path)

def run(stage):
    argv=['bash',linux_path(ROOT/'src/acdc_prefix_cache/run_wsl.sh'),'--stage',stage,'--out',linux_path(RESULTS)]
    if stage=='c':
        assert ALLOW_FULL_GRAPH, 'Stage C requires explicit full-graph authorization; it can run for hours.'
        argv+=['--allow-full-graph']
    if os.name=='nt':argv=['wsl','-d','Ubuntu','--']+argv
    print('Running:',argv)
    subprocess.run(argv,cwd=ROOT,check=True)


Campaign: H:\GenAI Research\sketch_prune\results\acdc_prefix_cache\20261019T173713Z
Defaults: GPT-2 Small, official MIB IOI, float32, train 16, validation 16, batch 2, seed 42, threshold 0.05.


## Stage A — correctness

The standard executor runs the complete transformer. The cached executor checks the descendant closure of **all** removed receivers against every skipped prefix sender, restores a clean residual boundary, seeds only proven-unaffected current senders, and executes the remaining native blocks. An earlier accepted deletion moves the start boundary earlier, even for a late candidate. Final residual/logit receivers conservatively retain the final block. Cache buffers are cloned before possible modification.

Stage A runs ten real small-TransformerLens unit tests, thirty single-edge queries across layers 0/2/4/6/8/10/11, eight dependent/random multi-edge states, a 32-edge restricted reverse-topological subsequence, a longer 64-edge prefix, checkpoint interruption/rebuild/replay, and held-out final-circuit checks. Official MIB and the corrected shared adapter supply independent oracles. Actual root forwards and block executions are counted; inference uses no gradients. Stage-A timings are diagnostic only.

In [6]:
if RUN_STAGE_A: run('a')
for name in ['correctness_results','stage_a_trajectories','stage_a_validation','runtime_estimates']:
    path=RESULTS/(name+'.csv')
    if path.exists():
        print(name);display(pd.read_csv(path))
    else:print(name+': not measured')
path=RESULTS/'decision_trace_comparison.csv'
if path.exists():
    trace=pd.read_csv(path)
    print('Saved decision comparison records:',len(trace))
    display(trace.groupby('trajectory')[['decision_matches','state_before_matches','state_after_matches','exact_loss_match']].all())


Running: ['wsl', '-d', 'Ubuntu', '--', 'bash', '/mnt/h/GenAI Research/sketch_prune/src/acdc_prefix_cache/run_wsl.sh', '--stage', 'a', '--out', '/mnt/h/GenAI Research/sketch_prune/results/acdc_prefix_cache/20261019T173713Z']
correctness_results


,test,case,examples,max_absolute_error,mean_absolute_error,exact_matches,official_max_absolute_error,passed
0,single,input->a0.h0<q>,16.0,0.0,0.0,16.0,0.000000,True
1,single,input->a0.h0<k>,16.0,0.0,0.0,16.0,0.000000,True
2,single,input->a0.h0<v>,16.0,0.0,0.0,16.0,0.000000,True
3,single,a0.h0->m0,16.0,0.0,0.0,16.0,0.000000,True
4,single,m1->a2.h0<q>,16.0,0.0,0.0,16.0,0.000000,True
5,single,m1->a2.h0<k>,16.0,0.0,0.0,16.0,0.000000,True
6,single,m1->a2.h0<v>,16.0,0.0,0.0,16.0,0.000000,True
7,single,a2.h0->m2,16.0,0.0,0.0,16.0,0.000000,True
8,single,m3->a4.h0<q>,16.0,0.0,0.0,16.0,0.000000,True
9,single,m3->a4.h0<k>,16.0,0.0,0.0,16.0,0.000000,True


stage_a_trajectories


,stage,trajectory,method,candidate_checks,accepted_removals,final_edge_count,circuit_hash,search_seconds_diagnostic_only,full_graph
0,A_correctness_not_performance,restricted_32,standard,32,29,32462,1096339b459593f6bb9c0f1ce343bee00773903989b9c7...,10.648042,False
1,A_correctness_not_performance,restricted_32,cached,32,29,32462,1096339b459593f6bb9c0f1ce343bee00773903989b9c7...,6.617505,False
2,A_correctness_not_performance,partial_prefix_64,standard,64,49,32442,8b491da4a5ca8c5d001b544333a419546ea8ab706be4ef...,19.355436,False
3,A_correctness_not_performance,partial_prefix_64,cached,64,49,32442,8b491da4a5ca8c5d001b544333a419546ea8ab706be4ef...,5.474827,False


stage_a_validation


,trajectory,examples,standard_loss,cached_loss,official_loss,max_absolute_error,official_max_absolute_error,retained_identity_match
0,restricted_32,16,-2.475981,-2.475981,-2.475981,0.0,0.000013,True
1,partial_prefix_64,16,-8.159912,-8.159912,-8.159912,0.0,0.000005,True


runtime_estimates


,discovery_examples,stage_b_medium_paired_seconds_estimate,stage_c_one_paired_full_seconds_estimate,stage_c_requested_repeats_seconds_estimate,performance_measurement,method
0,16,200.797842,13365.242485,40095.727455,False,"Stage-A diagnostic query timing, linear exampl..."
1,32,401.595684,26730.484970,80191.454909,False,"Stage-A diagnostic query timing, linear exampl..."
2,64,803.191368,53460.969939,160382.909818,False,"Stage-A diagnostic query timing, linear exampl..."
3,128,1606.382737,106921.939879,320765.819637,False,"Stage-A diagnostic query timing, linear exampl..."


Saved decision comparison records: 160


,decision_matches,state_before_matches,state_after_matches,exact_loss_match
trajectory,,,,
interrupt_rebuild_resume,True,True,True,True
partial_prefix_64,True,True,True,True
restricted_32,True,True,True,True


## Stage B — performance (not automatic)

Review the Stage-A report and runtime estimates before enabling this flag. This stage repeats seven-layer microbenchmarks and matched medium partial trajectories at 16/32/64/128 discovery examples. Both arms pay their own donor construction, tokenization and baseline/cache preparation. Timing synchronizes CUDA, alternates arm order, and records power/clock telemetry, cache bytes, isolated allocator peaks, total elapsed time, and held-out loss. Warmup, micro timings, search time, setup time, and validation are separate. Partial prefixes are late-layer-heavy and cannot establish full-search speedup.

In [7]:
if RUN_STAGE_B: run('b')
for name in ['layerwise_benchmark','full_acdc_benchmark','measured_paired_speedups']:
    path=RESULTS/(name+'.csv')
    if path.exists():
        frame=pd.read_csv(path);print(name+': '+str(len(frame))+' rows');display(frame.head(24))
    else:print(name+': not run')


Running: ['wsl', '-d', 'Ubuntu', '--', 'bash', '/mnt/h/GenAI Research/sketch_prune/src/acdc_prefix_cache/run_wsl.sh', '--stage', 'b', '--out', '/mnt/h/GenAI Research/sketch_prune/results/acdc_prefix_cache/20261019T173713Z']
layerwise_benchmark: 126 rows


,stage,method,trial,edge,earliest_affected_layer,seconds,mean_loss,max_absolute_error,cache_setup_seconds,cache_bytes,peak_allocated_mib,peak_reserved_mib
0,B_micro,standard,0,input->a0.h0<q>,0,0.289980,-2.903472,0.0,1.374156,145655808,962.535645,1094.0
1,B_micro,standard,0,input->a0.h0<k>,0,0.243105,-2.902653,0.0,1.374156,145655808,962.535645,1094.0
2,B_micro,standard,0,input->a0.h0<v>,0,0.313582,-2.900237,0.0,1.374156,145655808,962.535645,1094.0
3,B_micro,standard,0,m1->a2.h0<q>,2,0.262670,-2.896961,0.0,1.374156,145655808,962.535645,1094.0
4,B_micro,standard,0,m1->a2.h0<k>,2,0.291257,-2.903411,0.0,1.374156,145655808,962.535645,1094.0
5,B_micro,standard,0,m1->a2.h0<v>,2,0.216930,-2.895514,0.0,1.374156,145655808,962.535645,1094.0
6,B_micro,standard,0,m3->a4.h0<q>,4,0.198774,-2.912856,0.0,1.374156,145655808,962.535645,1094.0
7,B_micro,standard,0,m3->a4.h0<k>,4,0.216739,-2.902656,0.0,1.374156,145655808,962.535645,1094.0
8,B_micro,standard,0,m3->a4.h0<v>,4,0.231816,-2.902457,0.0,1.374156,145655808,962.535645,1094.0
9,B_micro,standard,0,m5->a6.h0<q>,6,0.214818,-2.919817,0.0,1.374156,145655808,962.535645,1094.0


full_acdc_benchmark: 50 rows


,stage,method,trial,discovery_examples,candidate_checks,total_seconds,actual_checks_this_attempt,full_graph,cache_preparation_seconds,warmup_seconds_excluded,...,peak_reserved_mib,cache_bytes,accepted_removals,final_edge_count,circuit_hash,validation_loss,seconds_per_check,resumed,success,previous_attempt_wall_seconds
0,C,standard,0,16,32491,10272.195205,32491.0,True,1.531914,0.774325,...,1094.0,145655808.0,32313.0,178.0,d40691854bab970d2b9fca5f61ff510d35355778b23186...,-11.429369,0.316108,False,True,0.0
1,C,cached,0,16,32491,5962.982873,32491.0,True,0.750970,0.165629,...,1252.0,302444544.0,32313.0,178.0,d40691854bab970d2b9fca5f61ff510d35355778b23186...,-11.429369,0.183504,False,True,0.0
2,B,standard,0,16,64,19.525842,64.0,False,0.543526,0.757672,...,1094.0,145655808.0,49.0,32442.0,8b491da4a5ca8c5d001b544333a419546ea8ab706be4ef...,-8.159912,0.296599,False,True,0.0
3,B,cached,0,16,64,5.347759,64.0,False,0.553334,0.242652,...,1252.0,302444544.0,49.0,32442.0,8b491da4a5ca8c5d001b544333a419546ea8ab706be4ef...,-8.159912,0.074913,False,True,0.0
4,B,cached,1,16,64,5.411606,64.0,False,0.584820,0.202747,...,1252.0,302444544.0,49.0,32442.0,8b491da4a5ca8c5d001b544333a419546ea8ab706be4ef...,-8.159912,0.075419,False,True,0.0
5,B,standard,1,16,64,18.299424,64.0,False,0.616251,0.726209,...,1094.0,145655808.0,49.0,32442.0,8b491da4a5ca8c5d001b544333a419546ea8ab706be4ef...,-8.159912,0.276300,False,True,0.0
6,B,standard,2,16,64,19.225115,64.0,False,0.643557,0.769958,...,1094.0,145655808.0,49.0,32442.0,8b491da4a5ca8c5d001b544333a419546ea8ab706be4ef...,-8.159912,0.290337,False,True,0.0
7,B,cached,2,16,64,6.474974,64.0,False,0.974732,0.223272,...,1252.0,302444544.0,49.0,32442.0,8b491da4a5ca8c5d001b544333a419546ea8ab706be4ef...,-8.159912,0.085941,False,True,0.0
8,B,standard,0,16,128,34.553820,128.0,False,0.634046,0.785593,...,1094.0,145655808.0,113.0,32378.0,6a63a65d6503ca738f0b751d7b8ee743aa5a038025da79...,-8.191929,0.264998,False,True,0.0
9,B,cached,0,16,128,11.203174,128.0,False,0.782300,0.186307,...,1252.0,302444544.0,113.0,32378.0,6a63a65d6503ca738f0b751d7b8ee743aa5a038025da79...,-8.191929,0.081413,False,True,0.0


measured_paired_speedups: 25 rows


,stage,discovery_examples,candidate_checks,trial,cached,standard,speedup
0,B,16,64,0,5.347759,19.525842,3.651220
1,B,16,64,1,5.411606,18.299424,3.381515
2,B,16,64,2,6.474974,19.225115,2.969141
3,B,16,128,0,11.203174,34.553820,3.084289
4,B,16,128,1,10.921637,36.003016,3.296485
5,B,16,128,2,11.643277,33.635422,2.888828
6,B,32,64,0,10.142924,35.436849,3.493751
7,B,32,64,1,10.555464,36.321649,3.441028
8,B,32,64,2,9.598636,38.074550,3.966663
9,B,32,128,0,18.813128,65.880675,3.501846


## Stage C — full graph (disabled by default)

This stage requires passing A and B, plus explicit `ALLOW_FULL_GRAPH=True` and `RUN_STAGE_C=True`. It visits all 32,491 real edges with identical seeds, dataset, loss, threshold and traversal in both arms. Checkpoints save the actual retained identities and trace; rebuilding caches and replay overhead count toward cost. Do not start this automatically. The independent log's 2.35× speedup and 686-edge circuit are not targets for a different task loss/data implementation.

In [9]:
if RUN_STAGE_C: run('c')
for path in sorted(RESULTS.glob('*.png')):
    print(path.name);display(Image(filename=str(path)))
report=RESULTS/'ACDC_PREFIX_CACHE_REPORT.md'
if report.exists(): display(Markdown(report.read_text(encoding='utf-8')))
else: print('No conclusions: Stage A has not produced a report yet.')


Running: ['wsl', '-d', 'Ubuntu', '--', 'bash', '/mnt/h/GenAI Research/sketch_prune/src/acdc_prefix_cache/run_wsl.sh', '--stage', 'c', '--out', '/mnt/h/GenAI Research/sketch_prune/results/acdc_prefix_cache/20261019T173713Z', '--allow-full-graph']


KeyboardInterrupt: 